<a href="https://colab.research.google.com/github/Johnogunlola/MRes-AI/blob/MRes/5_Models_evaluation_PAX_Demand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas numpy matplotlib seaborn scikit-learn statsmodels xgboost lightgbm catboost prophet tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.7 MB/s eta 0:00:00


In [ ]:
import os, re, json, argparse, warnings, math, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

In [ ]:
SEED = 42
np.random.seed(SEED)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom[denom == 0] = 1.0
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom[denom == 0] = 1.0
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

# -----------------------------
# Safe imports (optional models)
# -----------------------------
HAVE_XGB, HAVE_LGB, HAVE_CAT, HAVE_PROPhet, HAVE_TF = False, False, False, False, False

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    pass

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    HAVE_LGB = True
except Exception:
    pass

try:
    from catboost import CatBoostRegressor
    HAVE_CAT = True
except Exception:
    pass

try:
    # Prophet >= 1.0
    from prophet import Prophet
    HAVE_PROPhet = True
except Exception:
    try:
        # legacy fbprophet
        from fbprophet import Prophet
        HAVE_PROPhet = True
    except Exception:
        pass

try:
    import tensorflow as tf
    from tensorflow import keras
    HAVE_TF = True
except Exception:
    pass

from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
def parse_yyyymm(x):
    """Convert integer/float/string yyyymm to pandas Timestamp at month start."""
    if pd.isna(x): return pd.NaT
    s = str(x)
    s = re.sub(r'\.0$', '', s)  # handle 201501.0
    if len(s) == 6 and s.isdigit():
        return pd.Period(s, freq='M').to_timestamp()  # month start
    return pd.NaT

def load_and_clean(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)
    df.columns = [re.sub(r'\s+', ' ', c).strip().lower() for c in df.columns]

    required = [
        'this_period', 'group_name', 'airport_1_name', 'airport_2_name',
        'total_pax_this_period', 'total_pax_scheduled_this_period', 'total_pax_charter_this_period'
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for c in ['total_pax_this_period', 'total_pax_scheduled_this_period', 'total_pax_charter_this_period']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df['date'] = df['this_period'].apply(parse_yyyymm)
    df['route'] = df['airport_1_name'].str.strip() + ' → ' + df['airport_2_name'].str.strip()

    # keep essentials
    df = df[['date', 'route', 'total_pax_this_period']].dropna(subset=['date', 'route'])
    df = df.rename(columns={'total_pax_this_period': 'pax'})
    df = df.sort_values(['route', 'date'])

    return df

In [ ]:
def build_route_series(df: pd.DataFrame, route: str) -> pd.Series:
    """Monthly continuous index from first to last appearance; fill missing with 0 (inactive)."""
    s = df[df['route'] == route][['date', 'pax']].drop_duplicates()
    s = s.set_index('date').sort_index()

    # make monthly frequency
    idx = pd.date_range(s.index.min(), s.index.max(), freq='MS')
    s = s.reindex(idx)
    s['pax'] = s['pax'].fillna(0)
    s = s['pax']
    s.name = route
    return s

def longest_active_span(y: pd.Series):
    """Return longest consecutive span (start, end) where y>0, and its length in months."""
    active = (y.values > 0).astype(int)
    best_len, best_start = 0, None
    cur_len, cur_start = 0, None
    for i, a in enumerate(active):
        if a == 1:
            if cur_len == 0:
                cur_start = i
            cur_len += 1
            if cur_len > best_len:
                best_len = cur_len
                best_start = cur_start
        else:
            cur_len, cur_start = 0, None
    if best_len == 0:
        return None, None, 0
    start = y.index[best_start]
    end = y.index[best_start + best_len - 1]
    return start, end, best_len

def select_top_routes(df: pd.DataFrame, top_n: int = 10, min_months: int = 36):
    """Pick top_n routes by longest active continuity, breaking ties with total pax in that span."""
    routes = df['route'].unique()
    rows = []
    for r in routes:
        y = build_route_series(df, r)
        s, e, L = longest_active_span(y)
        if L < min_months:
            continue
        span = y.loc[s:e]
        rows.append({
            'route': r,
            'span_start': s,
            'span_end': e,
            'months': L,
            'total_pax_in_span': float(span.sum())
        })
    if not rows:
        raise ValueError("No route meets the minimum months criterion.")
    info = pd.DataFrame(rows).sort_values(['months', 'total_pax_in_span'], ascending=[False, False])
    return info.head(top_n)

In [ ]:
def split_train_test(y: pd.Series, span_start, span_end, test_horizon=12):
    """Restrict to active span; use last H months as test, remainder train."""
    ys = y.loc[span_start:span_end].copy()
    if len(ys) <= test_horizon + 12:
        # ensure enough training history
        raise ValueError("Span too short for split; need at least H+12 months.")
    train = ys.iloc[:-test_horizon]
    test = ys.iloc[-test_horizon:]
    return train, test

In [ ]:
def seasonal_naive(train: pd.Series, H=12, s=12):
    # forecast equals last season
    fc = train.shift(s).iloc[-H:]
    # if initial portion missing, backfill
    if len(fc) < H:
        fc = pd.Series([np.nan] * H, index=train.index[-H:])
    fc = fc.fillna(method='bfill')
    return fc

In [ ]:
def sarimax_best(train: pd.Series, H=12):
    pdq_grid = [(1,1,1), (2,1,2), (1,0,1), (0,1,1)]
    PDQ_grid = [(1,1,1,12), (0,1,1,12), (1,0,1,12), (1,1,0,12)]
    best_aic, best_model = np.inf, None
    best_order, best_seasonal = None, None
    for p,d,q in pdq_grid:
        for P,D,Q,s in PDQ_grid:
            try:
                m = SARIMAX(train, order=(p,d,q),
                            seasonal_order=(P,D,Q,s),
                            enforce_stationarity=False,
                            enforce_invertibility=False)
                fit = m.fit(disp=False)
                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best_model = fit
                    best_order = (p,d,q)
                    best_seasonal = (P,D,Q,s)
            except Exception:
                continue
    if best_model is None:
        # fallback
        best_model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12),
                             enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        best_order = (1,1,1)
        best_seasonal = (1,1,1,12)
    fc = best_model.forecast(steps=H)
    return fc, {'order': best_order, 'seasonal_order': best_seasonal, 'aic': best_aic}

In [ ]:
def make_features(y: pd.Series):
    df_feat = pd.DataFrame({'y': y})
    for L in range(1, 13):
        df_feat[f'lag_{L}'] = df_feat['y'].shift(L)
    df_feat['roll3'] = df_feat['y'].rolling(3).mean()
    df_feat['roll6'] = df_feat['y'].rolling(6).mean()
    df_feat['roll12'] = df_feat['y'].rolling(12).mean()
    df_feat['month'] = df_feat.index.month
    df_feat = df_feat.dropna()
    return df_feat

def split_features_for_test(df_feat, H=12):
    X = df_feat.drop(columns=['y'])
    y = df_feat['y']
    return X.iloc[:-H], X.iloc[-H:], y.iloc[:-H], y.iloc[-H:]

In [ ]:
def xgboost_forecast(y: pd.Series, H=12):
    if not HAVE_XGB:
        return None, {'error': 'xgboost not available'}
    df_feat = make_features(y)
    X_train, X_test, y_train, y_test = split_features_for_test(df_feat, H=H)
    model = XGBRegressor(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED
    )
    model.fit(X_train, y_train)
    y_pred = pd.Series(model.predict(X_test), index=y_test.index)
    info = {'n_estimators': 600, 'learning_rate': 0.05, 'max_depth': 6}
    return y_pred, info

def lightgbm_forecast(y: pd.Series, H=12):
    if not HAVE_LGB:
        return None, {'error': 'lightgbm not available'}
    df_feat = make_features(y)
    X_train, X_test, y_train, y_test = split_features_for_test(df_feat, H=H)
    model = LGBMRegressor(
        n_estimators=600, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        max_depth=-1, random_state=SEED
    )
    model.fit(X_train, y_train)
    y_pred = pd.Series(model.predict(X_test), index=y_test.index)
    info = {'n_estimators': 600, 'learning_rate': 0.05}
    return y_pred, info

def catboost_forecast(y: pd.Series, H=12):
    if not HAVE_CAT:
        return None, {'error': 'catboost not available'}
    df_feat = make_features(y)
    X_train, X_test, y_train, y_test = split_features_for_test(df_feat, H=H)
    cat_features = [X_train.columns.get_loc('month')]
    model = CatBoostRegressor(
        iterations=600, learning_rate=0.05, depth=6, loss_function='RMSE',
        random_seed=SEED, verbose=False
    )
    model.fit(X_train, y_train, cat_features=cat_features)
    y_pred = pd.Series(model.predict(X_test), index=y_test.index)
    info = {'iterations': 600, 'learning_rate': 0.05, 'depth': 6}
    return y_pred, info

In [ ]:
def prophet_forecast(train: pd.Series, H=12):
    if not HAVE_PROPhet:
        return None, {'error': 'prophet not available'}
    dfp = pd.DataFrame({'ds': train.index, 'y': train.values})
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(dfp)
    future = m.make_future_dataframe(periods=H, freq='MS')
    fc = m.predict(future).set_index('ds')['yhat'].iloc[-H:]
    return fc, {'yearly_seasonality': True}

In [ ]:
def lstm_forecast(y: pd.Series, H=12, window=12):
    if not HAVE_TF:
        return None, {'error': 'tensorflow not available'}
    arr = y.values.astype('float32')
    scale = arr.mean() if arr.mean() != 0 else 1.0
    arr_scaled = arr / scale

    # Build supervised sequences: window -> next value
    X_list, y_list, idx_list = [], [], []
    for i in range(window, len(arr_scaled)):
        X_list.append(arr_scaled[i-window:i])
        y_list.append(arr_scaled[i])
        idx_list.append(y.index[i])
    X_all = np.array(X_list)  # shape [T, window]
    y_all = np.array(y_list)

    # Split last H targets for test
    X_train, X_test = X_all[:-H], X_all[-H:]
    y_train, y_test = y_all[:-H], y_all[-H:]

    model = keras.Sequential([
        keras.layers.Input(shape=(window, 1)),
        keras.layers.LSTM(32, return_sequences=False),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss='mse')
    model.fit(X_train[..., None], y_train, epochs=50, batch_size=32, verbose=0)
    y_pred = model.predict(X_test[..., None], verbose=0).flatten() * scale
    pred = pd.Series(y_pred, index=y.index[-H:])
    return pred, {'window': window, 'epochs': 50, 'units': 32}

In [ ]:
def plot_forecast_overlay(route, train, test, preds_dict, outdir):
    """Overlay test actual vs all model forecasts; save PNG."""
    plt.figure(figsize=(10, 6))
    # context: last 24 months of train
    ctx = train.iloc[-24:] if len(train) > 24 else train
    plt.plot(ctx.index, ctx.values, label='Train (last 24m)', color='grey', alpha=0.7)
    plt.plot(test.index, test.values, label='Test Actual', color='black', linewidth=2)
    for model_name, series in preds_dict.items():
        if series is not None:
            plt.plot(series.index, series.values, label=model_name)
    plt.title(f"{route} — Forecast Overlay (Test Horizon={len(test)})")
    plt.xlabel("Month")
    plt.ylabel("Passengers")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(outdir, f"{safe_filename(route)}__overlay.png")
    plt.savefig(path, dpi=150)
    plt.close()
    return path

def plot_aggregate_bars(metrics_df, outdir):
    """Aggregate RMSE across routes by model and plot bar chart."""
    agg = metrics_df.groupby('model')['RMSE'].mean().sort_values()
    plt.figure(figsize=(10, 6))
    sns.barplot(x=agg.index, y=agg.values, color='#4C78A8')
    plt.title("Aggregate RMSE by Model (mean across selected routes)")
    plt.ylabel("RMSE (mean)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    path = os.path.join(outdir, "aggregate_rmse_bar.png")
    plt.savefig(path, dpi=150)
    plt.close()
    return path

def plot_heatmap(metrics_df, outdir):
    """Heatmap of RMSE per model per route."""
    pivot = metrics_df.pivot(index='route', columns='model', values='RMSE')
    plt.figure(figsize=(12, max(6, 0.5 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap='viridis')
    plt.title("RMSE per Model per Route")
    plt.xlabel("Model")
    plt.ylabel("Route")
    plt.tight_layout()
    path = os.path.join(outdir, "heatmap_rmse.png")
    plt.savefig(path, dpi=150)
    plt.close()
    return path

def safe_filename(s: str):
    return re.sub(r'[^a-zA-Z0-9_]+', '_', s.strip())

In [ ]:
def evaluate_route(route, y, span_start, span_end, H, outdir):
    train, test = split_train_test(y, span_start, span_end, test_horizon=H)
    preds = {}
    meta = {}

    # Seasonal Naive
    sn_pred = seasonal_naive(train, H=H, s=12)
    sn_pred.index = test.index
    preds['Seasonal Naïve (s=12)'] = sn_pred
    meta['Seasonal Naïve (s=12)'] = {'seasonality': 12}

    # SARIMAX
    sar_pred, sar_info = sarimax_best(train, H=H)
    sar_pred.index = test.index
    preds['SARIMAX'] = sar_pred
    meta['SARIMAX'] = sar_info

    # Prophet
    pr_pred, pr_info = prophet_forecast(train, H=H)
    preds['Prophet'] = pr_pred
    meta['Prophet'] = pr_info

    # XGBoost
    xgb_pred, xgb_info = xgboost_forecast(y.loc[span_start:span_end], H=H)
    preds['XGBoost'] = xgb_pred
    meta['XGBoost'] = xgb_info

    # LightGBM
    lgb_pred, lgb_info = lightgbm_forecast(y.loc[span_start:span_end], H=H)
    preds['LightGBM'] = lgb_pred
    meta['LightGBM'] = lgb_info

    # CatBoost
    cat_pred, cat_info = catboost_forecast(y.loc[span_start:span_end], H=H)
    preds['CatBoost'] = cat_pred
    meta['CatBoost'] = cat_info

    # LSTM
    lstm_pred, lstm_info = lstm_forecast(y.loc[span_start:span_end], H=H, window=12)
    preds['LSTM'] = lstm_pred
    meta['LSTM'] = lstm_info

    # Metrics & forecast results rows
    metrics_rows = []
    forecast_rows = []
    for model_name, y_pred in preds.items():
        if y_pred is None:
            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                'error': meta.get(model_name, {}).get('error', 'unavailable')
            })
            continue
        mae = mean_absolute_error(test, y_pred)
        r = rmse(test, y_pred)
        s = smape(test.values, y_pred.values)
        metrics_rows.append({
            'route': route, 'model': model_name,
            'MAE': mae, 'RMSE': r, 'sMAPE%': s
        })
        for d, a, p in zip(test.index, test.values, y_pred.values):
            forecast_rows.append({
                'route': route, 'date': d.strftime('%Y-%m-%d'),
                'actual': float(a), 'predicted': float(p),
                'model': model_name, 'split': 'test'
            })

    # Plot overlay
    overlay_path = plot_forecast_overlay(route, train, test, preds, outdir)

    return metrics_rows, forecast_rows, meta, overlay_path


In [ ]:
def main(args):
    outdir = args.outdir
    os.makedirs(outdir, exist_ok=True)

    df = load_and_clean(args.csv)
    # Select top routes by continuity
    info = select_top_routes(df, top_n=args.top_n, min_months=max(args.test_horizon + 24, 36))

    # Per-route evaluation
    all_metrics = []
    all_forecasts = []
    per_route_meta = {}
    per_route_plots = []

    for _, row in info.iterrows():
        route = row['route']
        span_start, span_end = row['span_start'], row['span_end']
        y = build_route_series(df, route)
        try:
            metrics_rows, forecast_rows, meta, plot_path = evaluate_route(
                route, y, span_start, span_end, H=args.test_horizon, outdir=outdir
            )
            all_metrics.extend(metrics_rows)
            all_forecasts.extend(forecast_rows)
            per_route_meta[route] = {
                'span_start': span_start.strftime('%Y-%m-%d'),
                'span_end': span_end.strftime('%Y-%m-%d'),
                'months': int(row['months']),
                'total_pax_in_span': float(row['total_pax_in_span']),
                'models': meta
            }
            per_route_plots.append({'route': route, 'overlay_plot': plot_path})
        except Exception as e:
            all_metrics.append({'route': route, 'model': 'ALL', 'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan, 'error': str(e)})

    # Save metrics & forecasts
    metrics_df = pd.DataFrame(all_metrics)
    forecasts_df = pd.DataFrame(all_forecasts)

    metrics_path = os.path.join(outdir, "model_metrics_by_route.csv")
    forecasts_path = os.path.join(outdir, "forecast_results.csv")
    metrics_df.to_csv(metrics_path, index=False)
    forecasts_df.to_csv(forecasts_path, index=False)

    # Aggregate metrics (mean across the selected routes)
    agg_df = metrics_df.dropna(subset=['RMSE']).groupby('model', as_index=False).agg({
        'MAE': 'mean', 'RMSE': 'mean', 'sMAPE%': 'mean'
    }).sort_values('RMSE')
    agg_path = os.path.join(outdir, "model_metrics_summary.csv")
    agg_df.to_csv(agg_path, index=False)

    # Global visuals
    agg_bar_path = plot_aggregate_bars(metrics_df.dropna(subset=['RMSE']), outdir)
    heatmap_path = plot_heatmap(metrics_df.dropna(subset=['RMSE']), outdir)

    # Manifest JSON
    manifest = {
        'timestamp_utc': dt.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        'data_source': os.path.abspath(args.csv),
        'top_n_routes': int(args.top_n),
        'test_horizon_months': int(args.test_horizon),
        'selected_routes': info.assign(
            span_start=info['span_start'].dt.strftime('%Y-%m-%d'),
            span_end=info['span_end'].dt.strftime('%Y-%m-%d')
        ).to_dict(orient='records'),
        'packages_available': {
            'xgboost': HAVE_XGB, 'lightgbm': HAVE_LGB, 'catboost': HAVE_CAT,
            'prophet': HAVE_PROPhet, 'tensorflow': HAVE_TF
        },
        'artifacts': {
            'metrics_by_route_csv': metrics_path,
            'forecast_results_csv': forecasts_path,
            'aggregate_metrics_csv': agg_path,
            'aggregate_rmse_bar_png': agg_bar_path,
            'heatmap_rmse_png': heatmap_path,
            'per_route_overlay_plots': per_route_plots
        },
        'per_route_model_meta': per_route_meta
    }
    manifest_path = os.path.join(outdir, "analysis_manifest.json")
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    # Console summary
    print("\n=== Aggregate (mean across routes) ===")
    print(agg_df.to_string(index=False))
    print(f"\nSaved:\n- Metrics by route: {metrics_path}\n- Forecasts: {forecasts_path}\n"
          f"- Aggregate metrics: {agg_path}\n- Manifest: {manifest_path}\n"
          f"- Plots: {agg_bar_path}, {heatmap_path}, plus per-route overlays.\n")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="UK Domestic Air Pax Forecasting: top routes by continuity")
    parser.add_argument("--csv", type=str, required=True, help="Path to the monthly CSV file")
    parser.add_argument("--top_n", type=int, default=10, help="Number of major routes to evaluate")
    parser.add_argument("--test_horizon", type=int, default=12, help="Forecast horizon in months")
    parser.add_argument("--outdir", type=str, default="outputs", help="Directory to save outputs")

    # Manually create args for Colab execution since command-line arguments are not passed automatically
    args = parser.parse_args(["--csv", "/content/Monthly Domestic Air Pax Route Analysis by Each Reporting Airport 2015 - 2025.csv"])
    main(args)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000947 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 74330.218750
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 46252.125000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 41117.906250
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000068 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 14502.020833
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

ValueError: zero-size array to reduction operation fmin which has no identity

<Figure size 1200x600 with 0 Axes>

# Task
Modify the `main` function to add a conditional check for an empty `metrics_df` (after dropping rows with NaN RMSE values) before calling `plot_aggregate_bars` and `plot_heatmap`. If `metrics_df` is empty, skip the plotting functions and print an informative message to the user. Also, investigate why all RMSE values might be NaN for all routes, as this indicates a potential issue in the `evaluate_route` function or `select_top_routes`.

## Check for empty metrics_df before plotting

### Subtask:
Add a conditional check in the `main` function to ensure `metrics_df.dropna(subset=['RMSE'])` is not empty before attempting to generate `plot_aggregate_bars` and `plot_heatmap`. If it's empty, skip plotting and inform the user.


**Reasoning**:
The subtask requires modifying the `main` function to include a conditional check before generating plots. I will replace the existing `main` function with an updated version that incorporates this check, ensuring `metrics_df.dropna(subset=['RMSE'])` is not empty before plotting aggregate bars and the heatmap.



In [ ]:
def main(args):
    outdir = args.outdir
    os.makedirs(outdir, exist_ok=True)

    df = load_and_clean(args.csv)
    # Select top routes by continuity
    info = select_top_routes(df, top_n=args.top_n, min_months=max(args.test_horizon + 24, 36))

    # Per-route evaluation
    all_metrics = []
    all_forecasts = []
    per_route_meta = {}
    per_route_plots = []

    for _, row in info.iterrows():
        route = row['route']
        span_start, span_end = row['span_start'], row['span_end']
        y = build_route_series(df, route)
        try:
            metrics_rows, forecast_rows, meta, plot_path = evaluate_route(
                route, y, span_start, span_end, H=args.test_horizon, outdir=outdir
            )
            all_metrics.extend(metrics_rows)
            all_forecasts.extend(forecast_rows)
            per_route_meta[route] = {
                'span_start': span_start.strftime('%Y-%m-%d'),
                'span_end': span_end.strftime('%Y-%m-%d'),
                'months': int(row['months']),
                'total_pax_in_span': float(row['total_pax_in_span']),
                'models': meta
            }
            per_route_plots.append({'route': route, 'overlay_plot': plot_path})
        except Exception as e:
            all_metrics.append({'route': route, 'model': 'ALL', 'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan, 'error': str(e)})

    # Save metrics & forecasts
    metrics_df = pd.DataFrame(all_metrics)
    forecasts_df = pd.DataFrame(all_forecasts)

    metrics_path = os.path.join(outdir,

SyntaxError: incomplete input (ipython-input-891759429.py, line 40)

**Reasoning**:
The previous code block resulted in a `SyntaxError` due to an incomplete line. I will correct the `metrics_path` and `forecasts_path` definitions and then implement the conditional check for plotting as requested by the subtask.



In [ ]:
def main(args):
    outdir = args.outdir
    os.makedirs(outdir, exist_ok=True)

    df = load_and_clean(args.csv)
    # Select top routes by continuity
    info = select_top_routes(df, top_n=args.top_n, min_months=max(args.test_horizon + 24, 36))

    # Per-route evaluation
    all_metrics = []
    all_forecasts = []
    per_route_meta = {}
    per_route_plots = []

    for _, row in info.iterrows():
        route = row['route']
        span_start, span_end = row['span_start'], row['span_end']
        y = build_route_series(df, route)
        try:
            metrics_rows, forecast_rows, meta, plot_path = evaluate_route(
                route, y, span_start, span_end, H=args.test_horizon, outdir=outdir
            )
            all_metrics.extend(metrics_rows)
            all_forecasts.extend(forecast_rows)
            per_route_meta[route] = {
                'span_start': span_start.strftime('%Y-%m-%d'),
                'span_end': span_end.strftime('%Y-%m-%d'),
                'months': int(row['months']),
                'total_pax_in_span': float(row['total_pax_in_span']),
                'models': meta
            }
            per_route_plots.append({'route': route, 'overlay_plot': plot_path})
        except Exception as e:
            all_metrics.append({'route': route, 'model': 'ALL', 'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan, 'error': str(e)})

    # Save metrics & forecasts
    metrics_df = pd.DataFrame(all_metrics)
    forecasts_df = pd.DataFrame(all_forecasts)

    metrics_path = os.path.join(outdir, "model_metrics_by_route.csv")
    forecasts_path = os.path.join(outdir, "forecast_results.csv")
    metrics_df.to_csv(metrics_path, index=False)
    forecasts_df.to_csv(forecasts_path, index=False)

    # Aggregate metrics (mean across the selected routes)
    agg_df = metrics_df.dropna(subset=['RMSE']).groupby('model', as_index=False).agg({
        'MAE': 'mean', 'RMSE': 'mean', 'sMAPE%': 'mean'
    }).sort_values('RMSE')
    agg_path = os.path.join(outdir, "model_metrics_summary.csv")
    agg_df.to_csv(agg_path, index=False)

    # Global visuals
    cleaned_metrics_df = metrics_df.dropna(subset=['RMSE'])
    if not cleaned_metrics_df.empty:
        agg_bar_path = plot_aggregate_bars(cleaned_metrics_df, outdir)
        heatmap_path = plot_heatmap(cleaned_metrics_df, outdir)
    else:
        print("\nNo valid RMSE values found across routes for plotting aggregate bars and heatmap. Skipping these plots.")
        agg_bar_path = "N/A"
        heatmap_path = "N/A"

    # Manifest JSON
    manifest = {
        'timestamp_utc': dt.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        'data_source': os.path.abspath(args.csv),
        'top_n_routes': int(args.top_n),
        'test_horizon_months': int(args.test_horizon),
        'selected_routes': info.assign(
            span_start=info['span_start'].dt.strftime('%Y-%m-%d'),
            span_end=info['span_end'].dt.strftime('%Y-%m-%d')
        ).to_dict(orient='records'),
        'packages_available': {
            'xgboost': HAVE_XGB, 'lightgbm': HAVE_LGB, 'catboost': HAVE_CAT,
            'prophet': HAVE_PROPhet, 'tensorflow': HAVE_TF
        },
        'artifacts': {
            'metrics_by_route_csv': metrics_path,
            'forecast_results_csv': forecasts_path,
            'aggregate_metrics_csv': agg_path,
            'aggregate_rmse_bar_png': agg_bar_path,
            'heatmap_rmse_png': heatmap_path,
            'per_route_overlay_plots': per_route_plots
        },
        'per_route_model_meta': per_route_meta
    }
    manifest_path = os.path.join(outdir, "analysis_manifest.json")
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    # Console summary
    print("\n=== Aggregate (mean across routes) ===")
    print(agg_df.to_string(index=False))
    print(f"\nSaved:\n- Metrics by route: {metrics_path}\n- Forecasts: {forecasts_path}\n"
          f"- Aggregate metrics: {agg_path}\n- Manifest: {manifest_path}\n"
          f"- Plots: {agg_bar_path}, {heatmap_path}, plus per-route overlays.\n")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="UK Domestic Air Pax Forecasting: top routes by continuity")
    parser.add_argument("--csv", type=str, required=True, help="Path to the monthly CSV file")
    parser.add_argument("--top_n", type=int, default=10, help="Number of major routes to evaluate")
    parser.add_argument("--test_horizon", type=int, default=12, help="Forecast horizon in months")
    parser.add_argument("--outdir", type=str, default="outputs", help="Directory to save outputs")

    # Manually create args for Colab execution since command-line arguments are not passed automatically
    args = parser.parse_args(["--csv", "/content/Monthly Domestic Air Pax Route Analysis by Each Reporting Airport 2015 - 2025.csv"])
    main(args)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000095 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 74330.218750
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000051 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 14502.020833
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

**Reasoning**:
The previous code block resulted in a `SyntaxError` due to an incomplete line. I will regenerate the `main` function with the corrected file paths and the conditional plotting logic as specified in the subtask.



In [ ]:
def main(args):
    outdir = args.outdir
    os.makedirs(outdir, exist_ok=True)

    df = load_and_clean(args.csv)
    # Select top routes by continuity
    info = select_top_routes(df, top_n=args.top_n, min_months=max(args.test_horizon + 24, 36))

    # Per-route evaluation
    all_metrics = []
    all_forecasts = []
    per_route_meta = {}
    per_route_plots = []

    for _, row in info.iterrows():
        route = row['route']
        span_start, span_end = row['span_start'], row['span_end']
        y = build_route_series(df, route)
        try:
            metrics_rows, forecast_rows, meta, plot_path = evaluate_route(
                route, y, span_start, span_end, H=args.test_horizon, outdir=outdir
            )
            all_metrics.extend(metrics_rows)
            all_forecasts.extend(forecast_rows)
            per_route_meta[route] = {
                'span_start': span_start.strftime('%Y-%m-%d'),
                'span_end': span_end.strftime('%Y-%m-%d'),
                'months': int(row['months']),
                'total_pax_in_span': float(row['total_pax_in_span']),
                'models': meta
            }
            per_route_plots.append({'route': route, 'overlay_plot': plot_path})
        except Exception as e:
            all_metrics.append({'route': route, 'model': 'ALL', 'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan, 'error': str(e)})

    # Save metrics & forecasts
    metrics_df = pd.DataFrame(all_metrics)
    forecasts_df = pd.DataFrame(all_forecasts)

    metrics_path = os.path.join(outdir, "model_metrics_by_route.csv")
    forecasts_path = os.path.join(outdir, "forecast_results.csv")
    metrics_df.to_csv(metrics_path, index=False)
    forecasts_df.to_csv(forecasts_path, index=False)

    # Aggregate metrics (mean across the selected routes)
    agg_df = metrics_df.dropna(subset=['RMSE']).groupby('model', as_index=False).agg({
        'MAE': 'mean', 'RMSE': 'mean', 'sMAPE%': 'mean'
    }).sort_values('RMSE')
    agg_path = os.path.join(outdir, "model_metrics_summary.csv")
    agg_df.to_csv(agg_path, index=False)

    # Global visuals
    cleaned_metrics_df = metrics_df.dropna(subset=['RMSE'])
    if not cleaned_metrics_df.empty:
        agg_bar_path = plot_aggregate_bars(cleaned_metrics_df, outdir)
        heatmap_path = plot_heatmap(cleaned_metrics_df, outdir)
    else:
        print("\nNo valid RMSE values found across routes for plotting aggregate bars and heatmap. Skipping these plots.")
        agg_bar_path = "N/A"
        heatmap_path = "N/A"

    # Manifest JSON
    manifest = {
        'timestamp_utc': dt.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        'data_source': os.path.abspath(args.csv),
        'top_n_routes': int(args.top_n),
        'test_horizon_months': int(args.test_horizon),
        'selected_routes': info.assign(
            span_start=info['span_start'].dt.strftime('%Y-%m-%d'),
            span_end=info['span_end'].dt.strftime('%Y-%m-%d')
        ).to_dict(orient='records'),
        'packages_available': {
            'xgboost': HAVE_XGB, 'lightgbm': HAVE_LGB, 'catboost': HAVE_CAT,
            'prophet': HAVE_PROPhet, 'tensorflow': HAVE_TF
        },
        'artifacts': {
            'metrics_by_route_csv': metrics_path,
            'forecast_results_csv': forecasts_path,
            'aggregate_metrics_csv': agg_path,
            'aggregate_rmse_bar_png': agg_bar_path,
            'heatmap_rmse_png': heatmap_path,
            'per_route_overlay_plots': per_route_plots
        },
        'per_route_model_meta': per_route_meta
    }
    manifest_path = os.path.join(outdir, "analysis_manifest.json")
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    # Console summary
    print("\n=== Aggregate (mean across routes) ===")
    print(agg_df.to_string(index=False))
    print(f"\nSaved:\n- Metrics by route: {metrics_path}\n- Forecasts: {forecasts_path}\n"
          f"- Aggregate metrics: {agg_path}\n- Manifest: {manifest_path}\n"
          f"- Plots: {agg_bar_path}, {heatmap_path}, plus per-route overlays.\n")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="UK Domestic Air Pax Forecasting: top routes by continuity")
    parser.add_argument("--csv", type=str, required=True, help="Path to the monthly CSV file")
    parser.add_argument("--top_n", type=int, default=10, help="Number of major routes to evaluate")
    parser.add_argument("--test_horizon", type=int, default=12, help="Forecast horizon in months")
    parser.add_argument("--outdir", type=str, default="outputs", help="Directory to save outputs")

    # Manually create args for Colab execution since command-line arguments are not passed automatically
    args = parser.parse_args(["--csv", "/content/Monthly Domestic Air Pax Route Analysis by Each Reporting Airport 2015 - 2025.csv"])
    main(args)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000059 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 74330.218750
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000098 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 14502.020833
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

**Reasoning**:
The previous code block resulted in a `SyntaxError` due to an incomplete line. I will regenerate the `main` function with the corrected file paths and the conditional plotting logic as specified in the subtask.



In [ ]:
def main(args):
    outdir = args.outdir
    os.makedirs(outdir, exist_ok=True)

    df = load_and_clean(args.csv)
    # Select top routes by continuity
    info = select_top_routes(df, top_n=args.top_n, min_months=max(args.test_horizon + 24, 36))

    # Per-route evaluation
    all_metrics = []
    all_forecasts = []
    per_route_meta = {}
    per_route_plots = []

    for _, row in info.iterrows():
        route = row['route']
        span_start, span_end = row['span_start'], row['span_end']
        y = build_route_series(df, route)
        try:
            metrics_rows, forecast_rows, meta, plot_path = evaluate_route(
                route, y, span_start, span_end, H=args.test_horizon, outdir=outdir
            )
            all_metrics.extend(metrics_rows)
            all_forecasts.extend(forecast_rows)
            per_route_meta[route] = {
                'span_start': span_start.strftime('%Y-%m-%d'),
                'span_end': span_end.strftime('%Y-%m-%d'),
                'months': int(row['months']),
                'total_pax_in_span': float(row['total_pax_in_span']),
                'models': meta
            }
            per_route_plots.append({'route': route, 'overlay_plot': plot_path})
        except Exception as e:
            all_metrics.append({'route': route, 'model': 'ALL', 'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan, 'error': str(e)})

    # Save metrics & forecasts
    metrics_df = pd.DataFrame(all_metrics)
    forecasts_df = pd.DataFrame(all_forecasts)

    metrics_path = os.path.join(outdir, "model_metrics_by_route.csv")
    forecasts_path = os.path.join(outdir, "forecast_results.csv")
    metrics_df.to_csv(metrics_path, index=False)
    forecasts_df.to_csv(forecasts_path, index=False)

    # Aggregate metrics (mean across the selected routes)
    agg_df = metrics_df.dropna(subset=['RMSE']).groupby('model', as_index=False).agg({
        'MAE': 'mean', 'RMSE': 'mean', 'sMAPE%': 'mean'
    }).sort_values('RMSE')
    agg_path = os.path.join(outdir, "model_metrics_summary.csv")
    agg_df.to_csv(agg_path, index=False)

    # Global visuals
    cleaned_metrics_df = metrics_df.dropna(subset=['RMSE'])
    if not cleaned_metrics_df.empty:
        agg_bar_path = plot_aggregate_bars(cleaned_metrics_df, outdir)
        heatmap_path = plot_heatmap(cleaned_metrics_df, outdir)
    else:
        print("\nNo valid RMSE values found across routes for plotting aggregate bars and heatmap. Skipping these plots.")
        agg_bar_path = "N/A"
        heatmap_path = "N/A"

    # Manifest JSON
    manifest = {
        'timestamp_utc': dt.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        'data_source': os.path.abspath(args.csv),
        'top_n_routes': int(args.top_n),
        'test_horizon_months': int(args.test_horizon),
        'selected_routes': info.assign(
            span_start=info['span_start'].dt.strftime('%Y-%m-%d'),
            span_end=info['span_end'].dt.strftime('%Y-%m-%d')
        ).to_dict(orient='records'),
        'packages_available': {
            'xgboost': HAVE_XGB, 'lightgbm': HAVE_LGB, 'catboost': HAVE_CAT,
            'prophet': HAVE_PROPhet, 'tensorflow': HAVE_TF
        },
        'artifacts': {
            'metrics_by_route_csv': metrics_path,
            'forecast_results_csv': forecasts_path,
            'aggregate_metrics_csv': agg_path,
            'aggregate_rmse_bar_png': agg_bar_path,
            'heatmap_rmse_png': heatmap_path,
            'per_route_overlay_plots': per_route_plots
        },
        'per_route_model_meta': per_route_meta
    }
    manifest_path = os.path.join(outdir, "analysis_manifest.json")
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    # Console summary
    print("\n=== Aggregate (mean across routes) ===")
    print(agg_df.to_string(index=False))
    print(f"\nSaved:\n- Metrics by route: {metrics_path}\n- Forecasts: {forecasts_path}\n"
          f"- Aggregate metrics: {agg_path}\n- Manifest: {manifest_path}\n"
          f"- Plots: {agg_bar_path}, {heatmap_path}, plus per-route overlays.\n")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="UK Domestic Air Pax Forecasting: top routes by continuity")
    parser.add_argument("--csv", type=str, required=True, help="Path to the monthly CSV file")
    parser.add_argument("--top_n", type=int, default=10, help="Number of major routes to evaluate")
    parser.add_argument("--test_horizon", type=int, default=12, help="Forecast horizon in months")
    parser.add_argument("--outdir", type=str, default="outputs", help="Directory to save outputs")

    # Manually create args for Colab execution since command-line arguments are not passed automatically
    args = parser.parse_args(["--csv", "/content/Monthly Domestic Air Pax Route Analysis by Each Reporting Airport 2015 - 2025.csv"])
    main(args)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000076 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 74330.218750
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000064 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 14502.020833
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

**Reasoning**:
The previous runs indicated that no valid RMSE values were found for plotting, suggesting an issue with models generating NaN predictions. I will add debugging print statements to the `evaluate_route` function to inspect the `train`, `test` data, and the `y_pred` from each model, as well as any exceptions caught during forecasting, to diagnose why RMSE values are consistently NaN.



In [ ]:
def evaluate_route(route, y, span_start, span_end, H, outdir):
    train, test = split_train_test(y, span_start, span_end, test_horizon=H)
    preds = {}
    meta = {}
    metrics_rows = [] # Initialize metrics_rows here
    forecast_rows = [] # Initialize forecast_rows here

    print(f"\n--- Evaluating route: {route} ---")
    print(f"Train data length: {len(train)}, Test data length: {len(test)}")
    print(f"Test data head:\n{test.head()}")
    print(f"Test data sum: {test.sum()}")

    models_to_evaluate = [
        ('Seasonal Naïve (s=12)', seasonal_naive, {'s': 12}),
        ('SARIMAX', sarimax_best, {}),
        ('Prophet', prophet_forecast, {}),
        ('XGBoost', xgboost_forecast, {}),
        ('LightGBM', lightgbm_forecast, {}),
        ('CatBoost', catboost_forecast, {}),
        ('LSTM', lstm_forecast, {'window': 12})
    ]

    for model_name, forecast_func, func_args in models_to_evaluate:
        print(f"  Attempting model: {model_name}")
        y_pred = None
        info_dict = {'error': 'unavailable'}
        try:
            # For ML models, make_features takes y for both train and test.
            # For statistical models, only train is passed for fitting.
            if model_name in ['XGBoost', 'LightGBM', 'CatBoost', 'LSTM']:
                y_pred, info_dict = forecast_func(y.loc[span_start:span_end], H=H, **func_args)
            else:
                y_pred, info_dict = forecast_func(train, H=H, **func_args)

            if y_pred is None:
                print(f"    Model {model_name} returned None prediction (error: {info_dict.get('error', 'unknown')})")
                metrics_rows.append({
                    'route': route, 'model': model_name,
                    'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                    'error': info_dict.get('error', 'unavailable')
                })
                preds[model_name] = None
                meta[model_name] = info_dict
                continue

            # Ensure prediction index aligns with test set
            if not y_pred.index.equals(test.index):
                # This can happen if make_features drops initial rows due to lags/rolling means
                # Re-index y_pred to match test period, filling missing with NaN if necessary
                original_y_pred = y_pred
                y_pred = pd.Series(index=test.index, dtype=float)
                y_pred.loc[original_y_pred.index] = original_y_pred

            if y_pred.isnull().any() or not np.isfinite(y_pred).all() or len(y_pred.dropna()) == 0:
                print(f"    Model {model_name} generated NaN/inf predictions or empty series after alignment.")
                metrics_rows.append({
                    'route': route, 'model': model_name,
                    'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                    'error': 'NaN/Inf predictions or empty after alignment'
                })
                preds[model_name] = None
                meta[model_name] = info_dict
                continue

            preds[model_name] = y_pred
            meta[model_name] = info_dict

            mae = mean_absolute_error(test, y_pred)
            r = rmse(test, y_pred)
            s = smape(test.values, y_pred.values)

            print(f"    {model_name} - MAE: {mae:.2f}, RMSE: {r:.2f}, sMAPE%: {s:.2f}")

            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': mae, 'RMSE': r, 'sMAPE%': s
            })
            for d, a, p in zip(test.index, test.values, y_pred.values):
                forecast_rows.append({
                    'route': route, 'date': d.strftime('%Y-%m-%d'),
                    'actual': float(a), 'predicted': float(p),
                    'model': model_name, 'split': 'test'
                })

        except Exception as e:
            print(f"    Error during {model_name} forecasting for route {route}: {e}")
            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                'error': str(e)
            })
            preds[model_name] = None
            meta[model_name] = {'error': str(e)}

    # Plot overlay
    overlay_path = plot_forecast_overlay(route, train, test, preds, outdir)

    return metrics_rows, forecast_rows, meta, overlay_path

**Reasoning**:
The previous outputs indicated that all RMSE values were NaN, leading to empty dataframes for plotting. The `evaluate_route` function was updated in the last turn to include debugging print statements to diagnose this. I need to run the `main` function again with this updated `evaluate_route` to observe the console output and pinpoint the source of the NaN RMSE values. This will help identify which models are failing or producing invalid predictions.



In [ ]:
def main(args):
    outdir = args.outdir
    os.makedirs(outdir, exist_ok=True)

    df = load_and_clean(args.csv)
    # Select top routes by continuity
    info = select_top_routes(df, top_n=args.top_n, min_months=max(args.test_horizon + 24, 36))

    # Per-route evaluation
    all_metrics = []
    all_forecasts = []
    per_route_meta = {}
    per_route_plots = []

    for _, row in info.iterrows():
        route = row['route']
        span_start, span_end = row['span_start'], row['span_end']
        y = build_route_series(df, route)
        try:
            metrics_rows, forecast_rows, meta, plot_path = evaluate_route(
                route, y, span_start, span_end, H=args.test_horizon, outdir=outdir
            )
            all_metrics.extend(metrics_rows)
            all_forecasts.extend(forecast_rows)
            per_route_meta[route] = {
                'span_start': span_start.strftime('%Y-%m-%d'),
                'span_end': span_end.strftime('%Y-%m-%d'),
                'months': int(row['months']),
                'total_pax_in_span': float(row['total_pax_in_span']),
                'models': meta
            }
            per_route_plots.append({'route': route, 'overlay_plot': plot_path})
        except Exception as e:
            all_metrics.append({'route': route, 'model': 'ALL', 'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan, 'error': str(e)})

    # Save metrics & forecasts
    metrics_df = pd.DataFrame(all_metrics)
    forecasts_df = pd.DataFrame(all_forecasts)

    metrics_path = os.path.join(outdir, "model_metrics_by_route.csv")
    forecasts_path = os.path.join(outdir, "forecast_results.csv")
    metrics_df.to_csv(metrics_path, index=False)
    forecasts_df.to_csv(forecasts_path, index=False)

    # Aggregate metrics (mean across the selected routes)
    agg_df = metrics_df.dropna(subset=['RMSE']).groupby('model', as_index=False).agg({
        'MAE': 'mean', 'RMSE': 'mean', 'sMAPE%': 'mean'
    }).sort_values('RMSE')
    agg_path = os.path.join(outdir, "model_metrics_summary.csv")
    agg_df.to_csv(agg_path, index=False)

    # Global visuals
    cleaned_metrics_df = metrics_df.dropna(subset=['RMSE'])
    if not cleaned_metrics_df.empty:
        agg_bar_path = plot_aggregate_bars(cleaned_metrics_df, outdir)
        heatmap_path = plot_heatmap(cleaned_metrics_df, outdir)
    else:
        print("\nNo valid RMSE values found across routes for plotting aggregate bars and heatmap. Skipping these plots.")
        agg_bar_path = "N/A"
        heatmap_path = "N/A"

    # Manifest JSON
    manifest = {
        'timestamp_utc': dt.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        'data_source': os.path.abspath(args.csv),
        'top_n_routes': int(args.top_n),
        'test_horizon_months': int(args.test_horizon),
        'selected_routes': info.assign(
            span_start=info['span_start'].dt.strftime('%Y-%m-%d'),
            span_end=info['span_end'].dt.strftime('%Y-%m-%d')
        ).to_dict(orient='records'),
        'packages_available': {
            'xgboost': HAVE_XGB, 'lightgbm': HAVE_LGB, 'catboost': HAVE_CAT,
            'prophet': HAVE_PROPhet, 'tensorflow': HAVE_TF
        },
        'artifacts': {
            'metrics_by_route_csv': metrics_path,
            'forecast_results_csv': forecasts_path,
            'aggregate_metrics_csv': agg_path,
            'aggregate_rmse_bar_png': agg_bar_path,
            'heatmap_rmse_png': heatmap_path,
            'per_route_overlay_plots': per_route_plots
        },
        'per_route_model_meta': per_route_meta
    }
    manifest_path = os.path.join(outdir, "analysis_manifest.json")
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    # Console summary
    print("\n=== Aggregate (mean across routes) ===")
    print(agg_df.to_string(index=False))
    print(f"\nSaved:\n- Metrics by route: {metrics_path}\n- Forecasts: {forecasts_path}\n"
          f"- Aggregate metrics: {agg_path}\n- Manifest: {manifest_path}\n"
          f"- Plots: {agg_bar_path}, {heatmap_path}, plus per-route overlays.\n")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="UK Domestic Air Pax Forecasting: top routes by continuity")
    parser.add_argument("--csv", type=str, required=True, help="Path to the monthly CSV file")
    parser.add_argument("--top_n", type=int, default=10, help="Number of major routes to evaluate")
    parser.add_argument("--test_horizon", type=int, default=12, help="Forecast horizon in months")
    parser.add_argument("--outdir", type=str, default="outputs", help="Directory to save outputs")

    # Manually create args for Colab execution since command-line arguments are not passed automatically
    args = parser.parse_args(["--csv", "/content/Monthly Domestic Air Pax Route Analysis by Each Reporting Airport 2015 - 2025.csv"])
    main(args)


def evaluate_route(route, y, span_start, span_end, H, outdir):
    train, test = split_train_test(y, span_start, span_end, test_horizon=H)
    preds = {}
    meta = {}
    metrics_rows = [] # Initialize metrics_rows here
    forecast_rows = [] # Initialize forecast_rows here

    print(f"\n--- Evaluating route: {route} ---")
    print(f"Train data length: {len(train)}, Test data length: {len(test)}")
    print(f"Test data head:\n{test.head()}")
    print(f"Test data sum: {test.sum()}")

    models_to_evaluate = [
        ('Seasonal Naïve (s=12)', seasonal_naive, {'s': 12}),
        ('SARIMAX', sarimax_best, {}),
        ('Prophet', prophet_forecast, {}),
        ('XGBoost', xgboost_forecast, {}),
        ('LightGBM', lightgbm_forecast, {}),
        ('CatBoost', catboost_forecast, {}),
        ('LSTM', lstm_forecast, {'window': 12})
    ]

    for model_name, forecast_func, func_args in models_to_evaluate:
        print(f"  Attempting model: {model_name}")
        y_pred = None
        info_dict = {'error': 'unavailable'}
        try:
            # For ML models, make_features takes y for both train and test.
            # For statistical models, only train is passed for fitting.
            if model_name in ['XGBoost', 'LightGBM', 'CatBoost', 'LSTM']:
                y_pred, info_dict = forecast_func(y.loc[span_start:span_end], H=H, **func_args)
            else:
                y_pred, info_dict = forecast_func(train, H=H, **func_args)

            if y_pred is None:
                print(f"    Model {model_name} returned None prediction (error: {info_dict.get('error', 'unknown')})")
                metrics_rows.append({
                    'route': route, 'model': model_name,
                    'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                    'error': info_dict.get('error', 'unavailable')
                })
                preds[model_name] = None
                meta[model_name] = info_dict
                continue

            # Ensure prediction index aligns with test set
            if not y_pred.index.equals(test.index):
                # This can happen if make_features drops initial rows due to lags/rolling means
                # Re-index y_pred to match test period, filling missing with NaN if necessary
                original_y_pred = y_pred
                y_pred = pd.Series(index=test.index, dtype=float)
                y_pred.loc[original_y_pred.index] = original_y_pred

            if y_pred.isnull().any() or not np.isfinite(y_pred).all() or len(y_pred.dropna()) == 0:
                print(f"    Model {model_name} generated NaN/inf predictions or empty series after alignment.")
                metrics_rows.append({
                    'route': route, 'model': model_name,
                    'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                    'error': 'NaN/Inf predictions or empty after alignment'
                })
                preds[model_name] = None
                meta[model_name] = info_dict
                continue

            preds[model_name] = y_pred
            meta[model_name] = info_dict

            mae = mean_absolute_error(test, y_pred)
            r = rmse(test, y_pred)
            s = smape(test.values, y_pred.values)

            print(f"    {model_name} - MAE: {mae:.2f}, RMSE: {r:.2f}, sMAPE%: {s:.2f}")

            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': mae, 'RMSE': r, 'sMAPE%': s
            })
            for d, a, p in zip(test.index, test.values, y_pred.values):
                forecast_rows.append({
                    'route': route, 'date': d.strftime('%Y-%m-%d'),
                    'actual': float(a), 'predicted': float(p),
                    'model': model_name, 'split': 'test'
                })

        except Exception as e:
            print(f"    Error during {model_name} forecasting for route {route}: {e}")
            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                'error': str(e)
            })
            preds[model_name] = None
            meta[model_name] = {'error': str(e)}

    # Plot overlay
    overlay_path = plot_forecast_overlay(route, train, test, preds, outdir)

    return metrics_rows, forecast_rows, meta, overlay_path


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


    Error during SARIMAX forecasting for route ISLE OF MAN → LIVERPOOL (JOHN LENNON): got an unexpected keyword argument 'squared'
  Attempting model: Prophet
    Error during Prophet forecasting for route ISLE OF MAN → LIVERPOOL (JOHN LENNON): got an unexpected keyword argument 'squared'
  Attempting model: XGBoost
    Error during XGBoost forecasting for route ISLE OF MAN → LIVERPOOL (JOHN LENNON): got an unexpected keyword argument 'squared'
  Attempting model: LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000078 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 14502.020833
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

**Reasoning**:
The previous run indicated two main issues: a `ValueError: too many values to unpack` for `seasonal_naive` (because it returned only one value but two were expected), and `TypeError: got an unexpected keyword argument 'squared'` for multiple forecasting models. The latter suggests a potential name conflict or incorrect passing of the `rmse` metric function's internal argument. To fix this, I will rename the custom `rmse` function to `calculate_rmse` to avoid any potential namespace conflicts, explicitly ensure `seasonal_naive` returns two values (forecast and info_dict), and update `evaluate_route` to use the new `calculate_rmse` function and correctly handle possible prediction index mismatches.



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def calculate_rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom[denom == 0] = 1.0
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

# -----------------------------
# Safe imports (optional models)
# -----------------------------
HAVE_XGB, HAVE_LGB, HAVE_CAT, HAVE_PROPhet, HAVE_TF = False, False, False, False, False

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    pass

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    HAVE_LGB = True
except Exception:
    pass

try:
    from catboost import CatBoostRegressor
    HAVE_CAT = True
except Exception:
    pass

try:
    # Prophet >= 1.0
    from prophet import Prophet
    HAVE_PROPhet = True
except Exception:
    try:
        # legacy fbprophet
        from fbprophet import Prophet
        HAVE_PROPhet = True
    except Exception:
        pass

try:
    import tensorflow as tf
    from tensorflow import keras
    HAVE_TF = True
except Exception:
    pass

from statsmodels.tsa.statespace.sarimax import SARIMAX

**Reasoning**:
The previous traceback indicated that the custom `rmse` function was conflicting with an internal keyword argument used by some of the forecasting models, leading to a `TypeError` (unexpected keyword argument 'squared'). Also, `seasonal_naive` was not returning enough values as expected. To fix this, I will directly use `mean_squared_error` from `sklearn.metrics` within the `evaluate_route` function instead of a custom `rmse` wrapper. I will also modify `seasonal_naive` to return both predictions and a dummy info dictionary to match the expected return signature in `evaluate_route`.



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom[denom == 0] = 1.0
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

# -----------------------------
# Safe imports (optional models)
# -----------------------------
HAVE_XGB, HAVE_LGB, HAVE_CAT, HAVE_PROPhet, HAVE_TF = False, False, False, False, False

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    pass

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    HAVE_LGB = True
except Exception:
    pass

try:
    from catboost import CatBoostRegressor
    HAVE_CAT = True
except Exception:
    pass

try:
    # Prophet >= 1.0
    from prophet import Prophet
    HAVE_PROPhet = True
except Exception:
    try:
        # legacy fbprophet
        from fbprophet import Prophet
        HAVE_PROPhet = True
    except Exception:
        pass

try:
    import tensorflow as tf
    from tensorflow import keras
    HAVE_TF = True
except Exception:
    pass

from statsmodels.tsa.statespace.sarimax import SARIMAX

def seasonal_naive(train: pd.Series, H=12, s=12):
    # forecast equals last season
    fc = train.shift(s).iloc[-H:]
    # if initial portion missing, backfill
    if len(fc) < H:
        fc = pd.Series([np.nan] * H, index=train.index[-H:])
    fc = fc.fillna(method='bfill')
    return fc, {'seasonality': s}


def evaluate_route(route, y, span_start, span_end, H, outdir):
    train, test = split_train_test(y, span_start, span_end, test_horizon=H)
    preds = {}
    meta = {}
    metrics_rows = [] # Initialize metrics_rows here
    forecast_rows = [] # Initialize forecast_rows here

    print(f"\n--- Evaluating route: {route} ---")
    print(f"Train data length: {len(train)}, Test data length: {len(test)}")
    print(f"Test data head:\n{test.head()}")
    print(f"Test data sum: {test.sum()}")

    models_to_evaluate = [
        ('Seasonal Naïve (s=12)', seasonal_naive, {'s': 12}),
        ('SARIMAX', sarimax_best, {}),
        ('Prophet', prophet_forecast, {}),
        ('XGBoost', xgboost_forecast, {}),
        ('LightGBM', lightgbm_forecast, {}),
        ('CatBoost', catboost_forecast, {}),
        ('LSTM', lstm_forecast, {'window': 12})
    ]

    for model_name, forecast_func, func_args in models_to_evaluate:
        print(f"  Attempting model: {model_name}")
        y_pred = None
        info_dict = {'error': 'unavailable'}
        try:
            # For ML models, make_features takes y for both train and test.
            # For statistical models, only train is passed for fitting.
            if model_name in ['XGBoost', 'LightGBM', 'CatBoost', 'LSTM']:
                y_pred, info_dict = forecast_func(y.loc[span_start:span_end], H=H, **func_args)
            else:
                y_pred, info_dict = forecast_func(train, H=H, **func_args)

            if y_pred is None:
                print(f"    Model {model_name} returned None prediction (error: {info_dict.get('error', 'unknown')})")
                metrics_rows.append({
                    'route': route, 'model': model_name,
                    'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                    'error': info_dict.get('error', 'unavailable')
                })
                preds[model_name] = None
                meta[model_name] = info_dict
                continue

            # Ensure prediction index aligns with test set
            if not y_pred.index.equals(test.index):
                # This can happen if make_features drops initial rows due to lags/rolling means
                # Re-index y_pred to match test period, filling missing with NaN if necessary
                original_y_pred = y_pred
                y_pred = pd.Series(index=test.index, dtype=float)
                y_pred.loc[original_y_pred.index] = original_y_pred

            if y_pred.isnull().any() or not np.isfinite(y_pred).all() or len(y_pred.dropna()) == 0:
                print(f"    Model {model_name} generated NaN/inf predictions or empty series after alignment.")
                metrics_rows.append({
                    'route': route, 'model': model_name,
                    'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                    'error': 'NaN/Inf predictions or empty after alignment'
                })
                preds[model_name] = None
                meta[model_name] = info_dict
                continue

            preds[model_name] = y_pred
            meta[model_name] = info_dict

            mae = mean_absolute_error(test, y_pred)
            r = mean_squared_error(test, y_pred, squared=False) # Directly use mean_squared_error
            s = smape(test.values, y_pred.values)

            print(f"    {model_name} - MAE: {mae:.2f}, RMSE: {r:.2f}, sMAPE%: {s:.2f}")

            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': mae, 'RMSE': r, 'sMAPE%': s
            })
            for d, a, p in zip(test.index, test.values, y_pred.values):
                forecast_rows.append({
                    'route': route, 'date': d.strftime('%Y-%m-%d'),
                    'actual': float(a), 'predicted': float(p),
                    'model': model_name, 'split': 'test'
                })

        except Exception as e:
            print(f"    Error during {model_name} forecasting for route {route}: {e}")
            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                'error': str(e)
            })
            preds[model_name] = None
            meta[model_name] = {'error': str(e)}

    # Plot overlay
    overlay_path = plot_forecast_overlay(route, train, test, preds, outdir)

    return metrics_rows, forecast_rows, meta, overlay_path


**Reasoning**:
The previous outputs indicated that `seasonal_naive` was not returning two values, causing a `ValueError`. Additionally, there was a `TypeError` due to a keyword argument conflict with the custom `rmse` function and internal model functions. I will modify `seasonal_naive` to return a dummy info dictionary, and replace all calls to the custom `rmse` function with `mean_squared_error` directly within `evaluate_route` to resolve these issues.



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom[denom == 0] = 1.0
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

# -----------------------------
# Safe imports (optional models)
# -----------------------------
HAVE_XGB, HAVE_LGB, HAVE_CAT, HAVE_PROPhet, HAVE_TF = False, False, False, False, False

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    pass

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    HAVE_LGB = True
except Exception:
    pass

try:
    from catboost import CatBoostRegressor
    HAVE_CAT = True
except Exception:
    pass

try:
    # Prophet >= 1.0
    from prophet import Prophet
    HAVE_PROPhet = True
except Exception:
    try:
        # legacy fbprophet
        from fbprophet import Prophet
        HAVE_PROPhet = True
    except Exception:
        pass

try:
    import tensorflow as tf
    from tensorflow import keras
    HAVE_TF = True
except Exception:
    pass

from statsmodels.tsa.statespace.sarimax import SARIMAX

def seasonal_naive(train: pd.Series, H=12, s=12):
    # forecast equals last season
    fc = train.shift(s).iloc[-H:]
    # if initial portion missing, backfill
    if len(fc) < H:
        fc = pd.Series([np.nan] * H, index=train.index[-H:])
    fc = fc.fillna(method='bfill')
    return fc, {'seasonality': s}


def evaluate_route(route, y, span_start, span_end, H, outdir):
    train, test = split_train_test(y, span_start, span_end, test_horizon=H)
    preds = {}
    meta = {}
    metrics_rows = [] # Initialize metrics_rows here
    forecast_rows = [] # Initialize forecast_rows here

    print(f"\n--- Evaluating route: {route} ---")
    print(f"Train data length: {len(train)}, Test data length: {len(test)}")
    print(f"Test data head:\n{test.head()}")
    print(f"Test data sum: {test.sum()}")

    models_to_evaluate = [
        ('Seasonal Naïve (s=12)', seasonal_naive, {'s': 12}),
        ('SARIMAX', sarimax_best, {}),
        ('Prophet', prophet_forecast, {}),
        ('XGBoost', xgboost_forecast, {}),
        ('LightGBM', lightgbm_forecast, {}),
        ('CatBoost', catboost_forecast, {}),
        ('LSTM', lstm_forecast, {'window': 12})
    ]

    for model_name, forecast_func, func_args in models_to_evaluate:
        print(f"  Attempting model: {model_name}")
        y_pred = None
        info_dict = {'error': 'unavailable'}
        try:
            # For ML models, make_features takes y for both train and test.
            # For statistical models, only train is passed for fitting.
            if model_name in ['XGBoost', 'LightGBM', 'CatBoost', 'LSTM']:
                y_pred, info_dict = forecast_func(y.loc[span_start:span_end], H=H, **func_args)
            else:
                y_pred, info_dict = forecast_func(train, H=H, **func_args)

            if y_pred is None:
                print(f"    Model {model_name} returned None prediction (error: {info_dict.get('error', 'unknown')})")
                metrics_rows.append({
                    'route': route, 'model': model_name,
                    'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                    'error': info_dict.get('error', 'unavailable')
                })
                preds[model_name] = None
                meta[model_name] = info_dict
                continue

            # Ensure prediction index aligns with test set
            if not y_pred.index.equals(test.index):
                # This can happen if make_features drops initial rows due to lags/rolling means
                # Re-index y_pred to match test period, filling missing with NaN if necessary
                original_y_pred = y_pred
                y_pred = pd.Series(index=test.index, dtype=float)
                y_pred.loc[original_y_pred.index] = original_y_pred

            if y_pred.isnull().any() or not np.isfinite(y_pred).all() or len(y_pred.dropna()) == 0:
                print(f"    Model {model_name} generated NaN/inf predictions or empty series after alignment.")
                metrics_rows.append({
                    'route': route, 'model': model_name,
                    'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                    'error': 'NaN/Inf predictions or empty after alignment'
                })
                preds[model_name] = None
                meta[model_name] = info_dict
                continue

            preds[model_name] = y_pred
            meta[model_name] = info_dict

            mae = mean_absolute_error(test, y_pred)
            r = mean_squared_error(test, y_pred, squared=False) # Directly use mean_squared_error
            s = smape(test.values, y_pred.values)

            print(f"    {model_name} - MAE: {mae:.2f}, RMSE: {r:.2f}, sMAPE%: {s:.2f}")

            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': mae, 'RMSE': r, 'sMAPE%': s
            })
            for d, a, p in zip(test.index, test.values, y_pred.values):
                forecast_rows.append({
                    'route': route, 'date': d.strftime('%Y-%m-%d'),
                    'actual': float(a), 'predicted': float(p),
                    'model': model_name, 'split': 'test'
                })

        except Exception as e:
            print(f"    Error during {model_name} forecasting for route {route}: {e}")
            metrics_rows.append({
                'route': route, 'model': model_name,
                'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan,
                'error': str(e)
            })
            preds[model_name] = None
            meta[model_name] = {'error': str(e)}

    # Plot overlay
    overlay_path = plot_forecast_overlay(route, train, test, preds, outdir)

    return metrics_rows, forecast_rows, meta, overlay_path

**Reasoning**:
The previous code block defined the corrected `seasonal_naive` and `evaluate_route` functions. Now, I need to redefine and execute the `main` function to ensure it utilizes these corrected functions and the updated plotting logic as per the subtask's requirements. This will run the full forecasting pipeline and should resolve the previous errors.



In [ ]:
def main(args):
    outdir = args.outdir
    os.makedirs(outdir, exist_ok=True)

    df = load_and_clean(args.csv)
    # Select top routes by continuity
    info = select_top_routes(df, top_n=args.top_n, min_months=max(args.test_horizon + 24, 36))

    # Per-route evaluation
    all_metrics = []
    all_forecasts = []
    per_route_meta = {}
    per_route_plots = []

    for _, row in info.iterrows():
        route = row['route']
        span_start, span_end = row['span_start'], row['span_end']
        y = build_route_series(df, route)
        try:
            metrics_rows, forecast_rows, meta, plot_path = evaluate_route(
                route, y, span_start, span_end, H=args.test_horizon, outdir=outdir
            )
            all_metrics.extend(metrics_rows)
            all_forecasts.extend(forecast_rows)
            per_route_meta[route] = {
                'span_start': span_start.strftime('%Y-%m-%d'),
                'span_end': span_end.strftime('%Y-%m-%d'),
                'months': int(row['months']),
                'total_pax_in_span': float(row['total_pax_in_span']),
                'models': meta
            }
            per_route_plots.append({'route': route, 'overlay_plot': plot_path})
        except Exception as e:
            all_metrics.append({'route': route, 'model': 'ALL', 'MAE': np.nan, 'RMSE': np.nan, 'sMAPE%': np.nan, 'error': str(e)})

    # Save metrics & forecasts
    metrics_df = pd.DataFrame(all_metrics)
    forecasts_df = pd.DataFrame(all_forecasts)

    metrics_path = os.path.join(outdir, "model_metrics_by_route.csv")
    forecasts_path = os.path.join(outdir, "forecast_results.csv")
    metrics_df.to_csv(metrics_path, index=False)
    forecasts_df.to_csv(forecasts_path, index=False)

    # Aggregate metrics (mean across the selected routes)
    agg_df = metrics_df.dropna(subset=['RMSE']).groupby('model', as_index=False).agg({
        'MAE': 'mean', 'RMSE': 'mean', 'sMAPE%': 'mean'
    }).sort_values('RMSE')
    agg_path = os.path.join(outdir, "model_metrics_summary.csv")
    agg_df.to_csv(agg_path, index=False)

    # Global visuals
    cleaned_metrics_df = metrics_df.dropna(subset=['RMSE'])
    if not cleaned_metrics_df.empty:
        agg_bar_path = plot_aggregate_bars(cleaned_metrics_df, outdir)
        heatmap_path = plot_heatmap(cleaned_metrics_df, outdir)
    else:
        print("\nNo valid RMSE values found across routes for plotting aggregate bars and heatmap. Skipping these plots.")
        agg_bar_path = "N/A"
        heatmap_path = "N/A"

    # Manifest JSON
    manifest = {
        'timestamp_utc': dt.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        'data_source': os.path.abspath(args.csv),
        'top_n_routes': int(args.top_n),
        'test_horizon_months': int(args.test_horizon),
        'selected_routes': info.assign(
            span_start=info['span_start'].dt.strftime('%Y-%m-%d'),
            span_end=info['span_end'].dt.strftime('%Y-%m-%d')
        ).to_dict(orient='records'),
        'packages_available': {
            'xgboost': HAVE_XGB, 'lightgbm': HAVE_LGB, 'catboost': HAVE_CAT,
            'prophet': HAVE_PROPhet, 'tensorflow': HAVE_TF
        },
        'artifacts': {
            'metrics_by_route_csv': metrics_path,
            'forecast_results_csv': forecasts_path,
            'aggregate_metrics_csv': agg_path,
            'aggregate_rmse_bar_png': agg_bar_path,
            'heatmap_rmse_png': heatmap_path,
            'per_route_overlay_plots': per_route_plots
        },
        'per_route_model_meta': per_route_meta
    }
    manifest_path = os.path.join(outdir, "analysis_manifest.json")
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    # Console summary
    print("\n=== Aggregate (mean across routes) ===")
    print(agg_df.to_string(index=False))
    print(f"\nSaved:\n- Metrics by route: {metrics_path}\n- Forecasts: {forecasts_path}\n"
          f"- Aggregate metrics: {agg_path}\n- Manifest: {manifest_path}\n"
          f"- Plots: {agg_bar_path}, {heatmap_path}, plus per-route overlays.\n")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="UK Domestic Air Pax Forecasting: top routes by continuity")
    parser.add_argument("--csv", type=str, required=True, help="Path to the monthly CSV file")
    parser.add_argument("--top_n", type=int, default=10, help="Number of major routes to evaluate")
    parser.add_argument("--test_horizon", type=int, default=12, help="Forecast horizon in months")
    parser.add_argument("--outdir", type=str, default="outputs", help="Directory to save outputs")

    # Manually create args for Colab execution since command-line arguments are not passed automatically
    args = parser.parse_args(["--csv", "/content/Monthly Domestic Air Pax Route Analysis by Each Reporting Airport 2015 - 2025.csv"])
    main(args)


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


    Error during SARIMAX forecasting for route ISLE OF MAN → LIVERPOOL (JOHN LENNON): got an unexpected keyword argument 'squared'
  Attempting model: Prophet
    Error during Prophet forecasting for route ISLE OF MAN → LIVERPOOL (JOHN LENNON): got an unexpected keyword argument 'squared'
  Attempting model: XGBoost
    Error during XGBoost forecasting for route ISLE OF MAN → LIVERPOOL (JOHN LENNON): got an unexpected keyword argument 'squared'
  Attempting model: LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000068 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 508
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 16
[LightGBM] [Info] Start training from score 14502.020833
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe